In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- 1. SETUP PATHS ---
base_dir = Path("/content/drive/MyDrive/Thesis/New_Results")

files = {
    #LSTM global
    "LSTM Global (1-Layer Log)": base_dir / "Subnets/LSTM_168_1_1layer_log/lstm_1layer_log_subnets_rolling_metrics.csv",
    "LSTM Global (3-Layer Log)": base_dir / "Subnets/LSTM_168_1_3layers_log/lstm_3layer_subnets_rolling_metrics.csv",
    "LSTM Global (1-Layer Raw)": base_dir / "Subnets/LSTM_168_1layer_RAW/lstm_1layer_subnets_rolling_metrics.csv",
    "LSTM Global (3-Layer Raw)": base_dir / "Subnets/LSTM_168_3layer_RAW/lstm_3layer_subnets_rolling_metrics.csv",
    #LSTM clustered
    "LSTM Clustered (1-Layer Log)": base_dir / "Subnets/Clustering/LSTM_1log_clusters/Cluster_Results_Rolling/lstm_1layer_log_clusters_rolling_metrics.csv",
    "LSTM Clustered (3-Layer Log)": base_dir / "Subnets/Clustering/LSTM_3log_clusters/Cluster_Results_Rolling/lstm_3layer_log_clusters_rolling_metrics.csv",
    "LSTM Clustered (1-Layer Raw)": base_dir / "Subnets/Clustering/LSTM_1layer_clusters/Cluster_Results_Rolling/lstm_1layer_clusters_rolling_metrics.csv",
    "LSTM Clustered (3-Layer Raw)": base_dir / "Subnets/Clustering/LSTM_3layers_clusters/Cluster_Results_Rolling/lstm_3layers_clusters_rolling_metrics.csv",

    #GRU
    "GRU Global (1-Layer Log)":  base_dir / "Subnets/GRU_1layer_log/gru_1layer_subnets_global_rolling_metrics.csv",
    "GRU Global (3-Layer Log)":  base_dir / "Subnets/GRU_3layer_log/gru_3layer_subnets_global_rolling_metrics.csv",
    "GRU Global (1-Layer Raw)":  base_dir / "Subnets/GRU_1layer_raw/gru_1layer_subnets_global_rolling_metrics.csv",
    "GRU Global (3-Layer Raw)":  base_dir / "Subnets/GRU_3layer_raw/gru_3layer_subnets_global_rolling_metrics.csv"

}

# --- 2. LOAD AND MERGE ---
all_results = []

print("Loading results...")
for model_name, file_path in files.items():
    if file_path.exists():
        df = pd.read_csv(file_path)

        # Standardize column names (Raw models might have 'R2_Scaled' instead of 'R2_LogScaled')
        if "R2_RawScaled" in df.columns:
            df = df.rename(columns={"R2_RawScaled": "R2", "RMSE_RawScaled": "RMSE"})
        elif "R2_LogScaled" in df.columns:
            df = df.rename(columns={"R2_LogScaled": "R2", "RMSE_LogScaled": "RMSE"})
        elif "R2_Scaled" in df.columns: # catch-all
            df = df.rename(columns={"R2_Scaled": "R2", "RMSE_Scaled": "RMSE"})

        # Add Model Label
        df["Model"] = model_name
        all_results.append(df)
        print(f"Loaded: {model_name} ({len(df)} subnets)")
    else:
        print(f"File not found: {file_path}")

if not all_results:
    print("No files loaded! Check your paths.")
else:
    # Combine into one Master DataFrame
    master_df = pd.concat(all_results, ignore_index=True)

    # --- 3. THE "MASTER TABLE" (Summary Stats) ---
    summary = master_df.groupby("Model")[["R2", "RMSE"]].agg(["mean", "median", "std"]).sort_values(by=("R2", "mean"), ascending=False)

    print("\n===MODEL COMPARISON LEADERBOARD===")
    print(summary)

    # Save the table for your thesis
    summary.to_csv(base_dir / "FINAL_MODEL_COMPARISON_TABLE.csv")

    # --- 4. THE "MONEY PLOT" (Boxplot) ---
    # This visualization proves statistical significance
    plt.figure(figsize=(14, 8))

    # Create Boxplot
    sns.boxplot(data=master_df, x="Model", y="R2", palette="viridis", showfliers=False) # showfliers=False hides extreme outliers for clarity

    plt.title("Comparison of Model Accuracy ($R^2$) Across All Subnets", fontsize=16)
    plt.ylabel("$R^2$ Score (Higher is Better)", fontsize=12)
    plt.xlabel("Model Architecture", fontsize=12)
    plt.xticks(rotation=15)
    plt.grid(True, axis='y', alpha=0.3)

    # Add a horizontal line at 0 (Baseline)
    plt.axhline(0, color='red', linestyle='--', linewidth=1, label="Baseline (Mean Predictor)")
    plt.legend()

    plt.tight_layout()
    plt.savefig(base_dir / "FINAL_MODEL_COMPARISON_PLOT.png")
    plt.show()

    print("\nComparison Plot Saved. Check the 'mean' R2 column above to pick your winner!")